In [ ]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

## Vectorized CurvesParamapAnalysis

Subclasses `CurvesParamapAnalysis` to override `compute_curves` with a vectorized version.
All window masks are pre-stacked into a single matrix; each frame is reduced to all window means
in one matrix multiply instead of looping over windows one at a time.

In [ ]:
import numpy as np
import copy
from tqdm import tqdm

from src.time_series_analysis.curves_paramap.framework import CurvesParamapAnalysis


class VectorizedCurvesParamapAnalysis(CurvesParamapAnalysis):

    def compute_curves(self):
        data = self.image_data.intensities_for_analysis
        is_3d = data.ndim == 4
        if not is_3d and data.ndim != 3:
            raise ValueError('Image data must be either 2D+time or 3D+time.')

        n_windows = len(self.windows)
        n_pixels = self.seg_data.seg_mask.size

        # Build (n_windows, n_pixels) boolean mask matrix — done once
        mask_matrix = np.zeros((n_windows, n_pixels), dtype=np.float32)
        for ix, window in enumerate(self.windows):
            m = np.zeros_like(self.seg_data.seg_mask, dtype=np.float32)
            if is_3d:
                ax_start, sag_start, cor_start, ax_end, sag_end, cor_end = window
                m[sag_start:sag_end+1, cor_start:cor_end+1, ax_start:ax_end+1] = 1
            else:
                ax_start, sag_start, ax_end, sag_end = window
                m[ax_start:ax_end+1, sag_start:sag_end+1] = 1
            mask_matrix[ix] = m.ravel()

        # Precompute per-window pixel counts for the mean
        window_sizes = mask_matrix.sum(axis=1, keepdims=True)  # (n_windows, 1)

        # Initialise curves list with window coordinate metadata
        self.curves = []
        for ix, window in enumerate(self.windows):
            entry = {}
            if is_3d:
                ax_start, sag_start, cor_start, ax_end, sag_end, cor_end = window
                entry['Window-Axial Start Pix'] = ax_start
                entry['Window-Sagittal Start Pix'] = sag_start
                entry['Window-Coronal Start Pix'] = cor_start
                entry['Window-Axial End Pix'] = ax_end
                entry['Window-Sagittal End Pix'] = sag_end
                entry['Window-Coronal End Pix'] = cor_end
            else:
                ax_start, sag_start, ax_end, sag_end = window
                entry['Window-Axial Start Pix'] = ax_start
                entry['Window-Sagittal Start Pix'] = sag_start
                entry['Window-Axial End Pix'] = ax_end
                entry['Window-Sagittal End Pix'] = sag_end
            entry['TIC'] = []
            self.curves.append(entry)

        # One matmul per frame → all window means at once
        n_frames = data.shape[3] if is_3d else data.shape[0]
        for frame_ix in tqdm(range(n_frames), desc='Computing curves'):
            frame = data[:, :, :, frame_ix] if is_3d else data[frame_ix]
            frame_flat = frame.ravel().astype(np.float32)           # (n_pixels,)
            means = (mask_matrix @ frame_flat) / window_sizes[:, 0] # (n_windows,)
            for ix, val in enumerate(means):
                self.curves[ix]['TIC'].append(float(val))

        if self.curves_output_path:
            self.save_curves()


print('VectorizedCurvesParamapAnalysis defined')

## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [ ]:
from src.image_loading.options import get_scan_loaders

print('Available scan loaders:', list(get_scan_loaders().keys()))

In [ ]:
scan_type = 'nifti'

scan_path = '/Users/samantha/Desktop/tul/china data/p34/new_v1/CEUS-26152-1.nii.gz'
scan_loader_kwargs = {
    'transpose': False,
}

In [ ]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [ ]:
from src.seg_loading.options import get_seg_loaders

print('Available segmentation loaders:', list(get_seg_loaders().keys()))

In [ ]:
seg_type = 'nifti'

seg_path = '/Users/samantha/Desktop/tul/china data/p34/new_v1/manual_vois/v1.1_necrotic_removed.nii.gz'
seg_loader_kwargs = {}

In [ ]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## CEUS Quantitative Temporal Curve Analysis (Parametric Map Mode — Parallel)

In [ ]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

_, all_analysis_funcs = get_analysis_types()
print('Available analysis functions:', list(all_analysis_funcs.keys()))

In [ ]:
analysis_funcs = ['tic']

# Set frame rate
image_data.frame_rate = 1

analysis_kwargs = {
    'ax_vox_ovrlp': 5.0,
    'sag_vox_ovrlp': 5.0,
    'cor_vox_ovrlp': 5.0,
    'ax_vox_len': 5.0,
    'sag_vox_len': 5.0,
    'cor_vox_len': 5.0,
}

In [ ]:
analyzed_image_data = copy.deepcopy(image_data)

analysis_obj = VectorizedCurvesParamapAnalysis(analyzed_image_data, seg_data, analysis_funcs, **analysis_kwargs)
analysis_obj.compute_curves()

print('Analysis object type:', type(analysis_obj))
print('Number of windows:', len(analysis_obj.windows))

## Curve Quantification

In [ ]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print('Available quantification functions:', quantification_funcs.keys())

In [ ]:
function_names = ['lognormal_fit_full']
output_path = '/Users/samantha/Desktop/tul/china data/p34/new_v1/manual_paramap/output.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
    'tic_name': 'TIC'
}

In [ ]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

print('curve_quant.analysis_objs type:', type(curve_quant.analysis_objs))

## Parametric Map Saving

In [ ]:
from src.entrypoints import visualization_step

vis_type = 'paramap'
params = []
vis_funcs = []
vis_kwargs = {
    'paramap_folder_path': '/Users/samantha/Desktop/tul/china data/p34/new_v1/manual_paramap',
    'hide_all_visualizations': False,
}

vis_obj = visualization_step(curve_quant, vis_type, params, vis_funcs, **vis_kwargs)